In [1]:
import pandas as pd
import time
import numpy as np

In [4]:
# Complete Dataset
curated_bow = pd.read_csv('code_test_no_noise.csv', sep=',')
curated_bow.dropna()
curated_bow.columns = [col.strip() for col in curated_bow.columns]
curated_bow

,id,line,contents,class_name,class_value,path,testName,verdict
0,0,0,"1,23 @@",build_status,1,0,0,0
1,0,0,This plugin downloads/installs Node and NPM lo...,build_status,1,0,0,0
2,0,0,"It's supposed to work on Windows, OS X and Linux.",build_status,1,0,0,0
3,0,0,NaN,build_status,1,0,0,0
4,0,0,# Installing,build_status,1,0,0,0
...,...,...,...,...,...,...,...,...
1554,0,0,"12,7 @@ import java.util.List;",build_status,0,0,0,0
1555,0,0,factory.getJspmRunner().execute(argume...,build_status,0,0,0,0
1556,0,0,"public final void execute(String args, Map...",build_status,0,0,0,0
1557,0,0,"61,7 @@ public final class EmberMojo extends A...",build_status,0,0,0,0


In [5]:
# Duplicate Columns Removed
curated_bow = curated_bow.loc[:,~curated_bow.columns.duplicated()].copy()
curated_bow

,id,line,contents,class_name,class_value,path,testName,verdict
0,0,0,"1,23 @@",build_status,1,0,0,0
1,0,0,This plugin downloads/installs Node and NPM lo...,build_status,1,0,0,0
2,0,0,"It's supposed to work on Windows, OS X and Linux.",build_status,1,0,0,0
3,0,0,NaN,build_status,1,0,0,0
4,0,0,# Installing,build_status,1,0,0,0
...,...,...,...,...,...,...,...,...
1554,0,0,"12,7 @@ import java.util.List;",build_status,0,0,0,0
1555,0,0,factory.getJspmRunner().execute(argume...,build_status,0,0,0,0
1556,0,0,"public final void execute(String args, Map...",build_status,0,0,0,0
1557,0,0,"61,7 @@ public final class EmberMojo extends A...",build_status,0,0,0,0


In [6]:
partitions_ = 1
bin_size = len(curated_bow.index)//partitions_ 
print(bin_size)

1559


In [11]:
def run_Panda():
    bins_counter = 0
    partition_values = pd.DataFrame()
    j_col= pd.DataFrame()
    k_col= pd.DataFrame()
    
    for j in range(len(curated_bow.columns)): 
        start_j = time.time()
        clustered_att = curated_bow.sort_values(by = curated_bow.columns[j])
        for k in range(len(curated_bow.columns)): 
            bins_counter = 0
            if k != j:
                for y in range(partitions_): 
                    
                    parition_mean = clustered_att.iloc[bins_counter: bins_counter + bin_size, j].values.mean()
                    partition_sd = clustered_att.iloc[bins_counter: bins_counter + bin_size, j].values.std()
                    mean_over_sd = parition_mean/partition_sd if partition_sd else 0

                    noise_score = pd.DataFrame([np.nan_to_num(abs(x - (mean_over_sd)))
                                                for x in clustered_att.iloc[bins_counter:bins_counter + bin_size, k]])

                    partition_values = pd.concat([partition_values, noise_score], axis=0)
                    
                    #dropping existing indices with the same name and replacing them with new, using reset index
                    partition_values= partition_values.reset_index(drop=True) 
                    bins_counter = bins_counter + bin_size
            else:
                continue
            
            k_col= pd.concat([k_col, partition_values], axis= 1)
            partition_values.drop(partition_values.index, inplace=True)
                
        
        j_col = pd.concat([j_col, k_col], axis = 1)
        end_j = time.time()
        print("column j: " + str(j))

        print(f"Runtime of the program for j {end_j - start_j}")
    return j_col

In [12]:
sss = run_Panda()
sss['max_noise'] = sss.max(axis=1)
s = sss.sort_values(by='max_noise', ascending = False)

UFuncTypeError: ufunc 'subtract' did not contain a loop with signature matching types (dtype('<U7'), dtype('float64')) -> None

In [ ]:
len(curated_bow.columns)

In [ ]:
print(s.head())